# Patch Time Series Transformer
> Revolutionary! 

In [ ]:
#| default_exp patchtst

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn as nn, torch.nn.functional as F, lightning.pytorch as pl, warnings, math
from typing import Optional
from torch import Tensor
from physiojepa.layers import Patch, MultiHeadAttention, get_activation_fn, tAPE, PositionalEncoding, PatchAugmentations
from physiojepa.tokenizers import TS_Tokenizer, TS_Tokenizer_Complex, InceptionTokenizer, PatchEncoder, MultiScaleTokenizer
from physiojepa.utils import trunc_normal_
from physiojepa.augmentations import unpatch
from physiojepa.loss import cosine_similarity_loss, mse_loss, mae_loss, huber_loss, mse_variance_loss
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts

In [ ]:
#| export
class TSTBlock(nn.Module):
    def __init__(self, 
                 d_model, # dimension of patch embeddings
                 n_heads, # number of attention heads per layer
                 d_ff=256, # dimension of feedforward layer in each transformer layer
                 attn_dropout=0, 
                 dropout=0., 
                 bias=True,
                 activation="gelu", 
                 pre_norm=False,
                 rotary_pes=False
                ):
        super().__init__()
        
        assert not d_model%n_heads, f"d_model ({d_model}) must be divisible by n_heads ({n_heads})"

        # Multi-Head attention
        self.self_attn = MultiHeadAttention(
            dim=d_model,
            num_heads=n_heads,
            qkv_bias=bias,
            qk_scale=None,
            attn_drop=attn_dropout,
            proj_drop=dropout,
            rotary_pes=rotary_pes
            )

        # Add & Norm
        self.dropout_attn = nn.Dropout(dropout) 
        self.norm_attn = nn.LayerNorm(d_model)

        # Position-wise Feed-Forward
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff, bias=bias), 
                                get_activation_fn(activation), # note do not put functions in sequential, it makes things non-deterministic
                                nn.Dropout(dropout),
                                nn.Linear(d_ff, d_model, bias=bias))

        # Add & Norm
        self.dropout_ffn = nn.Dropout(dropout)
        self.norm_ffn = nn.LayerNorm(d_model)

        self.pre_norm = pre_norm


    def forward(self, src:Tensor, mask:Optional[Tensor]=None):
        """
        src: tensor [bs x q_len x d_model]
        channel_mask: tensor [bs x n_channels]
        """
        # Multi-Head attention sublayer
        if self.pre_norm:
            src = self.norm_attn(src)
        ## Multi-Head attention
        src2 = self.self_attn(src, mask=mask)
        
        ## Add & Norm
        src = src + self.dropout_attn(src2) # Add: residual connection with residual dropout
        if not self.pre_norm:
            src = self.norm_attn(src)
        # Feed-forward sublayer
        if self.pre_norm:
            src = self.norm_ffn(src)
        ## Position-wise Feed-Forward

        src2 = self.ff(src)

        ## Add & Norm
        src = src + self.dropout_ffn(src2) # Add: residual connection with residual dropout
        if not self.pre_norm:
            src = self.norm_ffn(src)

        return src
    
#| export
class MaskedAutogressionFeedForward(nn.Module):
    def __init__(self, 
                 c_in, # the number of input channels
                 patch_len, # the length of the patches (either stft or interval length)
                 d_model, # the dimension of the initial linear layers for inputting patches into transformer
                 shared_recreation=True, # indicator of whether to project each channel individually or together
                 ):
        super().__init__()

        self.shared_recreation = shared_recreation
        self.n_vars = c_in
        self.patch_len = patch_len
        self.d_model = d_model
        self.c_in = c_in

        # Input encoding: projection of feature vectors onto a d-dim vector space
        ## note that this could be an MLP too, if you want
        if not shared_recreation:
            self.W_P = nn.ModuleList()
            for _ in range(self.n_vars): self.W_P.append(nn.Linear(d_model, patch_len))
        else:
            self.W_P = nn.Linear(d_model, patch_len)

    def forward(self, x) -> Tensor:          
        """
        input: x: tensor [bs x nvars x d_model x num_patch]
        returns: x: tensor [bs x num_patch x nvars x patch_len]
        """
        # Input embedding
        bs, C, D, P = x.shape
        x = x.permute(0,3,1,2) # [bs x num_patch x nvars x d_model]
        if not self.shared_recreation:
            x_out = []
            for c in range(self.n_vars): 
                ci = 0 if self.c_in != C and C == 1 else c # if there is only one channel (from conv), just use that one for each channel recreation
                z = self.W_P[c](x[:,:,ci,:])
                x_out.append(z)
            x = torch.stack(x_out, dim=2)
        else:
            x_out = []
            for c in range(self.n_vars):
                ci = 0 if self.c_in != C and C == 1 else c # if there is only one channel (from conv), just use that one for each channel recreation
                z = self.W_P(x[:,:,ci,:])
                x_out.append(z)
            x = torch.stack(x_out, dim=2)
        return x

In [ ]:
#| export
class PatchTFTSimple(nn.Module):
     def __init__(self,
                   c_in,
                    patch_size,
                    patch_stride,
                    num_patches,
                    d_model,
                    n_heads,
                    d_ff,
                    num_layers,
                    augmentations=['patch_mask', 'jitter_zero_mask', 'channel_masking'],
                    mask_ratio=0.1,
                    shared_embedding=False,
                    pretrain_head=True,
                    dropout=0.0,
                    attn_dropout=0.0,
                    act='gelu',
                    pre_norm=False,
                    pe_type='tAPE',
                    qkv_bias=True,
                    init_std=0.02,
                    tokenizer_type='simple',
                    tokenizer_kwargs={}
                  ):
          super().__init__()
          self.c_in = c_in
          self.patch_size = patch_size
          self.patch_stride = patch_stride
          self.d_model = d_model
          self.num_patches = num_patches
          self.n_heads = n_heads
          self.d_ff = d_ff
          self.num_layers = num_layers
          self.act = act
          self.init_std = init_std
          self.pe_type = pe_type.lower()
          self.tokenizer_type = tokenizer_type.lower()
          self.pretrain_head = pretrain_head
          self.num_patches = num_patches
          self.shared_embedding = shared_embedding

          # Building the tokenizer

          self.patch_layer = Patch(patch_len=patch_size, stride=patch_stride)
          if self.tokenizer_type in ['simple_conv', 'simple']:
               self.tokenizer = TS_Tokenizer(
               c_in=c_in,
               patch_size=patch_size,
               d_model=d_model * c_in if not shared_embedding else d_model,
               patch_stride=patch_stride,
               shared_embedding=self.shared_embedding
               )
          elif self.tokenizer_type in ['complex_conv', 'complex']:
               self.tokenizer = TS_Tokenizer_Complex(
               c_in=c_in,
               patch_size=patch_size,
               d_model=d_model
               )
          elif self.tokenizer_type == 'linear':
               self.tokenizer = PatchEncoder(c_in=c_in, 
                                             patch_len=patch_size, 
                                             d_model=d_model,
                                             shared_embedding=self.shared_embedding
                                             )
          elif self.tokenizer_type == 'inception':
               self.tokenizer = InceptionTokenizer(c_in=c_in, 
                                                  patch_size=patch_size,
                                                  d_model=d_model * c_in if not shared_embedding else d_model,
                                                  patch_stride=patch_stride,
                                                  shared_embedding=self.shared_embedding,
                                                  **tokenizer_kwargs
                                                  )
          elif self.tokenizer_type == 'multiscale':
               self.tokenizer = MultiScaleTokenizer(c_in=c_in,
                                                   patch_size=patch_size,
                                                   d_model=d_model * c_in if not shared_embedding else d_model,
                                                   patch_stride=patch_stride,
                                                   shared_embedding=self.shared_embedding,
                                                   **tokenizer_kwargs
                                                   )
          else:
               raise ValueError(f"Invalid tokenizer type: {tokenizer_type}. Valid options are: 'simple_conv', 'complex_conv', 'linear', 'inception', 'multiscale'")
          # Positional Encoding
          if self.pe_type == 'tape':
               self.pe = tAPE(d_model=self.d_model, seq_len=self.num_patches)
          elif self.pe_type == 'learned':
               self.pe = PositionalEncoding(num_patch=self.num_patches, d_model=self.d_model)
          elif self.pe_type == 'rotary':
               self.pe = nn.Identity()
          # residual dropout
          self.dropout = nn.Dropout(dropout)
          # time series transformer layers/Encoder
          self.layers = nn.ModuleList([TSTBlock(d_model=self.d_model, 
                                                n_heads=n_heads, 
                                                d_ff=d_ff, 
                                                attn_dropout=attn_dropout, 
                                                dropout=dropout, 
                                                bias=qkv_bias,
                                                activation=act, 
                                                pre_norm=pre_norm, 
                                                rotary_pes=self.pe_type == 'rotary') for _ in range(num_layers)])
          self.apply(self._init_weights)
          self._rescale_blocks()
          # Head
          self.mask = PatchAugmentations(augmentations=augmentations, patch_mask_ratio=mask_ratio, jitter_zero_mask_ratio=mask_ratio)
          if self.pretrain_head:
               self.head = MaskedAutogressionFeedForward(c_in = self.c_in, patch_len = self.patch_size, d_model = self.d_model, shared_recreation=self.shared_embedding)

     def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv1d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Conv2d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Conv3d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

     def _rescale_blocks(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))

        for layer_id, layer in enumerate(self.layers):
            rescale(layer.self_attn.proj.weight.data, layer_id + 1) # rescale the attention weights
            rescale(layer.ff[3].weight.data, layer_id + 1) # rescale the feedforward weights

     def forward(self, x):
          """
          input from ds is [bs x n_vars x max_seq_len]
          z: tensor [bs x num_patch x n_vars x patch_len]
          """
          # REVIN
          bs = x.size(0)
          
          seq_len = x.size(-1)
          x = self.patch_layer(x, constant_pad=True, constant_pad_value=0)
          Y_true = x.clone().detach()
          
          if self.training and self.pretrain_head:
               z = self.mask(x) # z: [bs x num_patch x n_vars x patch_len] not implementing padding mask for masking, it shouldnt matter expects: [bs x num_patch x n_vars x patch_len]
          else:
               z = x
          if (self.tokenizer_type != 'linear'):
               z = unpatch(z, seq_len, remove_padding=True) # unpatch lol
          z = self.tokenizer(z) # z: [bs x num_patch x n_vars x patch_len] pad with 0, same as input padded values

          if z.dim() == 3:
               z = z.unsqueeze(2) # z: [bs x num_patch x 1 x d_model]
          # MASKING
          # EMBEDDING
          z = z.transpose(1,2) # z: [bs x nvars x num_patch x d_model]
          # positional encoding
          transformer_c_in = z.size(1)
          z = torch.reshape(z, (bs * transformer_c_in, self.num_patches, self.d_model)) # u: [bs * nvars x num_patch x d_model]
          z = self.pe(z) # z: [bs * nvars x num_patch x d_model]
          # residual dropout
          z = self.dropout(z) # z: [bs * nvars x num_patch x d_model] 

          # encoder layers
          for mod in self.layers: 
               z = mod(z, mask=None) # z: [bs * n_vars x num_patch x d_model]
          z = torch.reshape(z, (-1, transformer_c_in, self.num_patches, self.d_model)) # z: [bs x nvars x num_patch x d_model]
          z = z.permute(0,1,3,2) # z: [bs x nvars x d_model x num_patch]
          if self.pretrain_head:
               Y_pred = self.head(z)
          # PRETRAIN HEAD
          if self.pretrain_head:
               return z, Y_pred, Y_true
          else:
               return z

In [ ]:
#| export
class PatchTFTSimpleLightning(pl.LightningModule):
    def __init__(self,
                 learning_rate,
                 train_size,
                 batch_size,
                 n_gpus,
                 metrics={},
                 loss_func='mse',
                 weight_decay=0.,
                 epochs=100,
                 use_weight_decay_scheduler=False,
                 final_weight_decay=0.4,
                 optimizer_type='AdamW',
                 scheduler_type='OneCycle',
                 huber_delta=None, # huber loss delta, not used otherwise
                 scheduler_kwargs={},
                 transforms=None,
                 **patchmeup_kwargs
                 ):
        super().__init__()
        assert loss_func.lower() in ['mae','mse','cosine','huber', 'mse_variance']
        self.scheduler_type = scheduler_type
        if self.scheduler_type is not None:
            assert self.scheduler_type.lower() in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.train_size = train_size
        self.n_gpus = n_gpus
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.epochs = epochs
        self.total_steps = int(self.ipe*self.epochs)
        self.metrics = metrics
        self.loss_func = loss_func.lower()
        self.huber_delta = huber_delta
        self.optimizer_type = optimizer_type
        self.weight_decay = weight_decay
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.final_weight_decay = final_weight_decay
        self.scheduler_kwargs = scheduler_kwargs
        self.transforms = transforms
        self.model = PatchTFTSimple(**patchmeup_kwargs)
        #self.model.compile(fullgraph=False)
    
    def forward(self, x):
        z, x_hat, y = self.model(x)
        return z, x_hat, y
        
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.transforms is not None:
            batch = self.transforms(batch)
        x, _ = batch
        z, x_hat, y = self.model(x)
        cos_loss_ = cosine_similarity_loss(x_hat, y).to(self.device)
        mae_loss_ = mae_loss(x_hat, y).to(self.device)
        huber_loss_ = huber_loss(x_hat, y, delta=self.huber_delta if self.huber_delta is not None else 1).to(self.device)
        mse_loss_ = mse_loss(x_hat, y).to(self.device)
        mse_variance_loss_ = mse_variance_loss(x_hat, y, z).to(self.device)
        loss = cos_loss_ if self.loss_func == 'cosine' else\
               mae_loss_ if self.loss_func == 'mae' else\
               huber_loss_ if self.loss_func == 'huber' else\
               mse_variance_loss_ if self.loss_func == 'mse_variance' else\
               mse_loss_
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('train_cos_loss', cos_loss_, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('train_mae_loss', mae_loss_, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('train_huber_loss', huber_loss_, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('train_mse_loss', mse_loss_, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('train_mse_variance_loss', mse_variance_loss_, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log_dict({f'train_{metric.__name__}':metric(x_hat, y) for metric in self.metrics}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        
        if batch_idx % 50 == 0:
            with torch.no_grad():
                # Check representation stats
                x_hat_d = x_hat.clone().detach()
                y_d = y.clone().detach()
                z_d = z.clone().detach() # z: [bs x nvars x d_model x num_patch] 
                
                pred_mean = x_hat_d.mean().item()
                pred_std = x_hat_d.std().item()
                y_mean = y_d.mean().item()
                y_std = y_d.std().item()

                bs, C, d_model, num_patches = z_d.shape
                sample_size = min(1000, num_patches)
                indices = torch.randperm(num_patches)[:sample_size] # random samples
                z_d = z_d.permute(0,3,1,2) # [bs, num_patches, C, d_model]

                z_d = z_d[:, indices, :, :]
                flat = z_d.view(bs * sample_size, C, d_model)
                for i in range(C):
                    norm = F.normalize(flat[:,i,:], dim=1)
                    similarity_matrix = norm @ norm.T
                    z_mean = z_d[:,:,i,:].mean().item()
                    z_std = z_d[:,:,i,:].std().item()
                    pred_pairwise_cos_sim = similarity_matrix[~torch.eye(bs*sample_size, dtype=torch.bool)].mean().item()
                    self.log(f'z_cos_sim_channel_{i}', pred_pairwise_cos_sim)
                    self.log(f'z_mean_channel_{i}', z_mean)
                    self.log(f'z_std_channel_{i}', z_std)

                self.log('pred_mean', pred_mean)
                self.log('pred_std', pred_std)
                self.log('y_mean', y_mean)
                self.log('y_std', y_std)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, _ = batch
        z, x_hat, y = self.model(x)
        cos_loss_ = cosine_similarity_loss(x_hat, y).to(self.device)
        mae_loss_ = mae_loss(x_hat, y).to(self.device)
        huber_loss_ = huber_loss(x_hat, y, delta=self.huber_delta if self.huber_delta is not None else 1).to(self.device)
        mse_loss_ = mse_loss(x_hat, y).to(self.device)
        mse_variance_loss_ = mse_variance_loss(x_hat, y, z).to(self.device)
        loss = cos_loss_ if self.loss_func == 'cosine' else\
               mae_loss_ if self.loss_func == 'mae' else\
               huber_loss_ if self.loss_func == 'huber' else\
               mse_variance_loss_ if self.loss_func == 'mse_variance' else\
               mse_loss_
        self.log("val_loss", loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('val_cos_loss', cos_loss_, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('val_mae_loss', mae_loss_, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('val_huber_loss', huber_loss_, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('val_mse_loss', mse_loss_, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('val_mse_variance_loss', mse_variance_loss_, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log_dict({f'val_{metric.__name__}':metric(x_hat, y) for metric in self.metrics}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
    
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x, _ = batch
        z, x_hat, y = self.model(x)
        return z
    
    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
            {
                'params': (p for n, p in self.model.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.model.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            }
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, epochs=self.epochs, steps_per_epoch=self.ipe, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs) # lr max is initial LR
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()